# VocEd Lab 8 — Convolutions and a Minimal CNN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emilsar/VocEd/blob/main/Labs/Lab8_convolutions_minimal_cnn.ipynb)

In Lab 7 we chose the kernels ourselves — a Gaussian blur, a Sobel edge, a Gabor stripe — and
fed their answers to a random forest.  The gain was small, and we were left asking: *were they
the right kernels?*

A **convolutional neural network (CNN)** answers that by learning its kernels from the labelled
images.  In this lab we:

- compute a convolution **by hand**, and check it against the library;
- build a tiny CNN in PyTorch — 5,187 numbers to learn;
- train it, look at the kernels it **learned**, and test it once.

**Textbook:** [Chapter 8, Part 1 — Foundations of Convolutional Neural Networks](https://obooks.tech/imageprocessing/chapter8.html#part-1-foundations-of-convolutional-neural-networks).

---

### ⚡ Use a GPU

Training is much faster on a GPU.  In Colab: **Runtime → Change runtime type → T4 GPU**, then
run the notebook from the top.  It also works on a CPU, just slower.

### How the puzzles work 🧩🤔

- **🧩 Code puzzles:** replace every `___` and run the cell — a ✅ means you got it.
- **🤔 Think puzzles:** answer in the ✏️ cell below the question.  There is no check: discuss
  your answer with your group.

## 0. Setup — Get the Data

The same images and split as Labs 3–7.

In [ ]:
!git clone https://github.com/emilsar/VocEd.git
%cd VocEd

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

X = np.stack([np.load(f"imagedata/X/{i}.npy") for i in range(200)])   # (200, 3, 256, 256)
y = np.stack([np.load(f"imagedata/y/{i}.npy") for i in range(200)])   # (200, 256, 256)

nuc_px   = (y == 2).sum(axis=(1, 2))
quartile = np.digitize(nuc_px, np.percentile(nuc_px, [25, 50, 75]))
trainval_idx, test_idx = train_test_split(np.arange(200), test_size=0.2, stratify=quartile, random_state=42)
train_idx, val_idx = train_test_split(trainval_idx, test_size=40, stratify=quartile[trainval_idx], random_state=42)
assert list(test_idx[:5]) == [121, 152, 81, 4, 141], "Not the same test images as Labs 2–7"

def dice_score(pred, target, label):
    pred_mask   = (pred   == label)
    target_mask = (target == label)
    denominator = pred_mask.sum() + target_mask.sum()
    if denominator == 0:
        return 1.0
    return 2 * (pred_mask & target_mask).sum() / denominator

def image_dice(pred, mask):
    return (dice_score(pred, mask, 1) + dice_score(pred, mask, 2)) / 2      # cytoplasm and nucleus

def nc_ratio(mask):
    return (mask == 2).sum() / max((mask == 1).sum(), 1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using:", device)

## 1. Convolution by Hand

A 3 × 3 **kernel** slides over the image.  At each position it multiplies the nine pixels under
it by its nine weights and adds them up: that sum is one pixel of the output.

### 🧩 Puzzle 1 — One output pixel

Here is a 5 × 5 image with a bright right half, and a kernel that responds to a change from left
to right.  Compute the output at the centre: the 3 × 3 patch around pixel (2, 2), times the
kernel, summed.

📖 Book: [§8.2.2 Convolution by Hand](https://obooks.tech/imageprocessing/chapter8.html#convolution-by-hand)

<details><summary><b>💡 Hint</b> (click to open)</summary>

`patch * kernel` multiplies the nine pairs; one more method adds them up.

</details>

In [ ]:
image = np.array([[0, 0, 0, 1, 1],
                  [0, 0, 0, 1, 1],
                  [0, 0, 0, 1, 1],
                  [0, 0, 0, 1, 1],
                  [0, 0, 0, 1, 1]], dtype=float)

kernel = np.array([[-1, 0, 1],
                   [-1, 0, 1],
                   [-1, 0, 1]], dtype=float)

patch = image[1:4, 1:4]                 # the 3 × 3 window centred on (2, 2)
centre = (patch * kernel).___
print("Patch:\n", patch)
print("Output at the centre:", centre)

# ── Check ──
assert centre == 3, "Multiply the nine pairs, then add them up"
print("✅ Puzzle 1 solved!")

Now the whole image: slide the window over every position where it fits.  We check the result
against `scipy.ndimage.correlate`.  (Deep-learning libraries do exactly this *without* flipping the
kernel; mathematicians call that *correlation* and keep the word convolution for the flipped
version.  See the book's note on kernel flipping.)

### 🧩 Puzzle 2 — How big is the output?

Without padding, a 3 × 3 window fits in fewer positions than the image has pixels.  How many
output rows does a 5-row image give?  A 256-row image?

<details><summary><b>💡 Hint</b> (click to open)</summary>

The window's centre cannot sit on the first or the last row.

</details>

In [ ]:
from scipy.ndimage import correlate

def conv_by_hand(image, kernel):
    H, W = image.shape
    out = np.zeros((H - 2, W - 2))
    for r in range(H - 2):
        for c in range(W - 2):
            out[r, c] = (image[r:r + 3, c:c + 3] * kernel).sum()
    return out

out = conv_by_hand(image, kernel)
print(out)

rows_from_5   = ___
rows_from_256 = ___

# ── Check ──
assert np.allclose(out, correlate(image, kernel)[1:-1, 1:-1]), "Same as the library, away from the border"
assert rows_from_5 == out.shape[0] and rows_from_256 == 256 - 2, "A 3 × 3 window loses one row at the top and one at the bottom"
print("✅ Puzzle 2 solved!")

That is why CNN layers use `padding=1`: a ring of zeros around the image lets the window sit on
every pixel, so a 256 × 256 image gives a 256 × 256 output — one prediction per pixel.

## 2. A Minimal CNN

Our network has three layers:

| Layer | What it does | Output |
|---|---|---|
| `conv1` | 16 kernels, each 3 × 3 over the 3 colours | 16 feature maps |
| `conv2` | 32 kernels, each 3 × 3 over the 16 maps | 32 feature maps |
| `conv3` | a 1 × 1 kernel: for every pixel, a logistic regression of its 32 values | 3 class scores |

Between layers, **ReLU** sets negative values to 0 — the bend that lets the network learn curved
boundaries, as in Lab 6's MLP.  Every kernel weight is learned.

📖 Book: [§8.2.4 A Minimal CNN in PyTorch](https://obooks.tech/imageprocessing/chapter8.html#sec-minimal-cnn)

### 🧩 Puzzle 3 — Count the numbers to learn

A layer with `n_in` input maps, `n_out` kernels of size 3 × 3 has `n_in × n_out × 9` weights and
`n_out` biases (one per kernel).

<details><summary><b>💡 Hint</b> (click to open)</summary>

For `conv2`: 16 input maps, 32 kernels.  How many biases?

</details>

In [ ]:
class MinimalCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,  16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32,  3, kernel_size=1)
        self.relu  = nn.ReLU()

    def forward(self, x):                      # x: (batch, 3, 256, 256)
        x = self.relu(self.conv1(x))           # → (batch, 16, 256, 256)
        x = self.relu(self.conv2(x))           # → (batch, 32, 256, 256)
        return self.conv3(x)                   # → (batch,  3, 256, 256): one score per class

torch.manual_seed(0)
model = MinimalCNN().to(device)

conv1_count = 3 * 16 * 9 + 16
conv2_count = 16 * 32 * 9 + ___
conv3_count = 32 * 3 * 1 + 3
print(f"conv1 {conv1_count} + conv2 {conv2_count} + conv3 {conv3_count} = {conv1_count + conv2_count + conv3_count}")

# ── Check ──
assert conv1_count + conv2_count + conv3_count == sum(p.numel() for p in model.parameters()), "Weights plus one bias per kernel"
print("✅ Puzzle 3 solved!")

### 🤔 Think 1 — What can one prediction see?

1. One output pixel of `conv1` depends on a 3 × 3 patch of the image.  One output pixel of
   `conv2` depends on a 3 × 3 patch of `conv1`'s maps.  How big a patch of the **image** does it
   depend on?  *(This is its **receptive field**.)*
2. A nucleus is 30–60 pixels across.  Can this network tell whether a dark pixel is in the middle of
   a big nucleus or is a lone grain?  Compare with Lab 7's `blur8`.
3. Lab 7's random forest used about 54,000 training *pixels*.  The CNN trains on 120 whole
   *images*.  Why can it use every pixel, where the forest had to sample?

📖 Book: [§8.3.2 What “Receptive Field” Means](https://obooks.tech/imageprocessing/chapter8.html#what-receptive-field-means)

✏️ **Your answer:** *(double-click this cell and type here)*

## 3. Training

PyTorch feeds the network **batches** of images through a `DataLoader`.  For each batch, training
is three steps:

1. **forward:** compute the class scores and the loss — cross-entropy, which is small when the
   true class gets a high score;
2. **backward:** compute how the loss changes with each of the 5,187 weights (its **gradient**);
3. **step:** move every weight a little in the direction that lowers the loss (Adam, learning rate
   0.001).

One pass over the 120 training images is an **epoch**.  The book trains for 5; we train for 25 and
score the validation images after every epoch, to see whether 5 was enough.  *(About a minute on a
GPU; about 8 minutes on a CPU.)*

### 🧩 Puzzle 4 — The backward step

📖 Book: [§8.2.7 Training and Evaluation](https://obooks.tech/imageprocessing/chapter8.html#sec-cnn-training)

<details><summary><b>💡 Hint</b> (click to open)</summary>

The method on the loss that computes all the gradients is named after the direction it runs in.

</details>

In [ ]:
class SegDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, k):
        i = self.indices[k]
        return torch.from_numpy(X[i]), torch.from_numpy(y[i].astype(np.int64))

def predict_masks(model, indices):
    model.eval()
    with torch.no_grad():
        images = torch.from_numpy(X[indices]).to(device)
        return model(images).argmax(dim=1).cpu().numpy()      # argmax over the 3 class scores

torch.manual_seed(0)
train_loader = DataLoader(SegDataset(train_idx), batch_size=8, shuffle=True)

loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

losses, val_dices = [], []
for epoch in range(25):
    model.train()
    total = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        loss = loss_fn(model(images), masks)       # 1. forward
        optimizer.zero_grad()
        loss.___()                          # 2. backward
        optimizer.step()                           # 3. step
        total += loss.item()
    losses.append(total / len(train_loader))
    val_dices.append(np.mean([image_dice(p, y[i]) for p, i in zip(predict_masks(model, val_idx), val_idx)]))
    print(f"epoch {epoch + 1:2d}: average loss {losses[-1]:.4f}   validation Dice {val_dices[-1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(1, 26), losses, 'o-')
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('training loss')
axes[1].plot(range(1, 26), val_dices, 'o-')
axes[1].axvline(5, color='gray', linestyle='--')
axes[1].axhline(0.812, color='red', linestyle=':')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('validation Dice (grey: 5 epochs; red: Lab 4)')
plt.show()

# ── Check ──
assert losses[-1] < losses[0], "The loss should go down"
print("✅ Puzzle 4 solved!")

### 🤔 Think 2 — When is training finished?

1. After 5 epochs (the grey line), was the training loss still falling fast?  What was the validation
   Dice then, and after 25?
2. Where does the validation curve level off?  Why should you decide how long to train by looking at
   *validation* Dice, not training loss?
3. Compare the final validation Dice with Lab 4's 0.812 (red).  A network that learns its own 5 × 5
   features ends up — where?

📖 Book: [§8.2.8 Tuning the MinimalCNN: Hyperparameters](https://obooks.tech/imageprocessing/chapter8.html#tuning-the-minimalcnn-hyperparameters)

✏️ **Your answer:** *(double-click this cell and type here)*

## 4. Looking Inside

Each of `conv1`'s 16 kernels has 3 × 3 weights for each of the 3 colours, so we can show it as a
tiny 3 × 3 colour picture.  Then we show what each kernel's map looks like on image 7.

📖 Book: [§8.4.1 Learned Kernels: What MinimalCNN Discovered](https://obooks.tech/imageprocessing/chapter8.html#learned-kernels-what-minimalcnn-discovered)

In [ ]:
W = model.conv1.weight.detach().cpu().numpy()          # (16, 3, 3, 3)
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for k, ax in enumerate(axes.flat):
    w = W[k].transpose(1, 2, 0)                        # (3, 3, 3): rows, columns, colour
    ax.imshow((w - w.min()) / (w.max() - w.min()))     # stretch to 0–1 to show it as colour
    ax.set_title(f'kernel {k}')
    ax.axis('off')
plt.suptitle('The 16 learned conv1 kernels')
plt.show()

with torch.no_grad():
    maps = torch.relu(model.conv1(torch.from_numpy(X[7:8]).to(device)))[0].cpu().numpy()   # (16, 256, 256)
fig, axes = plt.subplots(2, 8, figsize=(20, 5.5))
for k, ax in enumerate(axes.flat):
    ax.imshow(maps[k], cmap='gray')
    ax.set_title(f'map {k}')
    ax.axis('off')
plt.suptitle('conv1 feature maps for image 7')
plt.show()

### 🤔 Think 3 — Learned or hand-made?

1. Find a map that looks like one of Lab 7's hand-made features — a plain colour channel, a blur, an
   edge map.  Which?
2. Are some maps almost blank?  What does a kernel whose map is all zero after ReLU contribute?
3. Nobody told the network to build an edge detector or a "dark-and-blue" detector.  Where did its
   kernels come from?

📖 Book: [§8.4.1 Learned Kernels: What MinimalCNN Discovered](https://obooks.tech/imageprocessing/chapter8.html#learned-kernels-what-minimalcnn-discovered)

✏️ **Your answer:** *(double-click this cell and type here)*

## 5. The Final Test — Once

📖 Book: [§8.2.7 Training and Evaluation](https://obooks.tech/imageprocessing/chapter8.html#sec-cnn-training)

In [ ]:
test_preds = predict_masks(model, test_idx)
nc_true = np.array([nc_ratio(y[i]) for i in test_idx])
nc_pred = np.array([nc_ratio(p) for p in test_preds])

dice  = np.mean([image_dice(p, y[i]) for p, i in zip(test_preds, test_idx)])
r2    = 1 - np.sum((nc_pred - nc_true) ** 2) / np.sum((nc_true - nc_true.mean()) ** 2)
close = np.mean(np.abs(nc_pred - nc_true) < 0.05)

print(f"{'':28s} test Dice   N/C R²   N/C within ±0.05")
print(f"{'MinimalCNN (5,187 weights)':28s} {dice:.4f}    {r2:7.3f}   {close:.0%}")
print()
print("Earlier labs, on the same 40 test images:")
print("  Lab 4 specks, fill              0.772     -0.465   35%")

### 🤔 Think 4 — Small network, small window

1. How does the MinimalCNN compare with Lab 4 and Lab 7 — with a 5 × 5 view and 25 epochs?
2. Run the cell below to see its masks next to the truth.  Where does it still go wrong?  Does
   that fit what you said about its receptive field in Think 1?
3. To see a whole nucleus, a prediction must depend on a window about 50 pixels wide.  Stacking
   3 × 3 layers adds 2 pixels per layer.  How many layers would that take?  What could make a
   layer see further without more layers?

📖 Book: [§8.3.1 Why a Single-Scale CNN Isn't Enough](https://obooks.tech/imageprocessing/chapter8.html#why-a-single-scale-cnn-isnt-enough)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col, k in enumerate([0, 1, 2, 3]):
    axes[0, col].imshow(y[test_idx[k]], cmap='viridis', vmin=0, vmax=2)
    axes[0, col].set_title(f'test image {test_idx[k]}: truth')
    axes[1, col].imshow(test_preds[k], cmap='viridis', vmin=0, vmax=2)
    axes[1, col].set_title(f'CNN: Dice {image_dice(test_preds[k], y[test_idx[k]]):.3f}')
for ax in axes.flat:
    ax.axis('off')
plt.show()

✏️ **Your answer:** *(double-click this cell and type here)*

## Wrap-up

**Key takeaways:**
- **A convolution is a sliding weighted sum.**  Without padding the output shrinks by 2 for a 3 × 3
  kernel; `padding=1` keeps one output per pixel.
- **A CNN learns its kernels.**  Its 5,187 weights started random and were moved, step by step, by
  the gradients of the loss.  Some learned kernels resemble Lab 7's hand-made ones.
- **Training is forward, backward, step** — repeated for every batch, for several epochs.
- **The receptive field limits what a prediction can know.**  Two 3 × 3 layers see 5 × 5 pixels —
  far smaller than a nucleus.  Lab 9's U-Net sees much further.

---

### Group discussion 👥

Split the work, then share your numbers:

| Person | Change, then re-run Sections 2–3 (re-run from the model cell) |
|---|---|
| A | 50 epochs instead of 25 |
| B | `lr=3e-4` instead of `1e-3` |
| C | Add a third 3 × 3 layer (32 → 32) before `conv3` |

1. Which change helped most on validation?  Did any make things worse?
2. Person C's network sees 7 × 7.  Was the gain worth the extra computation?
3. If you could only change one thing, would you train longer or see further?  Why?